# CANDLE V3 — learn head on the receiver encoder

**FULL TRAINING: HOLD pending deeper real-data debugging.** Engineering smoke passed;
the only natural trial did not qualify. This notebook is retained for a later locked run.

Variant C: frozen receiver encoder, head-only updates from current donor and receiver BASE.
Donor data stays at its owner. The receiver aggregates compatible head deltas, builds
receiver-space router sketches and commits only after current CAL acceptance.
This notebook explicitly passes `--peer-teaching`; it does not copy donor FC2 rows.

## Kaggle setup

- Enable Internet and GPU. Use a fresh session and an empty OUTPUT_DIR.
- Attach `khoilv2005/100-clients`; nested dataset folders are resolved by the launcher.
- One GPU, AMP, seed 42, 100 clients, 6 tasks x 20 rounds, batch 2048, Adam lr 0.001.
- Same native V1 paper neighborhoods (xi=0.8), binary_cosine router and backbone as Direct FC2.
- Same original V1 roles: BASE 80%, FIT 8%, CAL 2%, validation 10% of original TRAIN.
  Each task uses only current data; no historical raw-data replay.
  The inherited native learner also holds out 10% within current BASE for local validation;
  teaching BASE comes from the resulting client training subset.
- After task finalization: at most 2 donors / 3 targets per receiver, 8 head aggregation
  rounds at lr 0.01; a separate donor-only control also runs 8 rounds. Backbone stays frozen
  within each teaching session. Current BASE/FIT/CAL budgets are unchanged.
- Acceptance: recall >=95%, observed FAR <=0.1%, negative break=0, gain over controls,
  at least 32 positive/negative CAL samples. Missing evidence never becomes PASS.
- Discovery policy `control_fit_preflight_v1`: reject impossible role-count budgets before
  model transmission; test BASE-built router/mask controls on FIT before training head;
  defer when the observed FIT routing ceiling is below 95%. These probes do not install a router.
  This is a versioned policy change: continuation from the earlier policy is rejected.
- Communication budget: 128 MiB/task including snapshots, updates, sketches and verification.
- No intermediate TEST. After T5/R19 and terminal consolidation: original full 34-class TEST
  on disjoint receiver shards; pooled BinarySelf primary, MulticlassSelf secondary.
- Round checkpoints compressed, ZIP per task, 12 GiB output guard. This is a fresh-run launcher;
  arbitrary round resume is not exposed. Source is cloned from pinned GitHub; no embedded bundle.

## Smoke evidence and comparison limits

28/28 structural tests and 84/84 native checks passed. Controlled synthetic install=1;
natural installs=0 in the two-client smoke. The insufficient-CAL native trial now stops at
metadata before model transmission or head training; the controlled fixture exercises 8 head rounds.
Exact task-boundary resume matched. No TEST was read. Full-run effectiveness remains unproven.
Two mature T2 checkpoint trials identified a router/mask registration gap and a routing ceiling
of only 11/256 on target FIT examples. They do not authorize a successful deployment.

This is the original Variant C profile, not a single-component ablation of Direct FC2:
C allows 2 donors/3 targets and announced-current-peer teaching edges, while the published
Direct FC2 profile allows 1 donor/1 target restricted to the actual native neighborhood.
The native training/data/TEST protocols are shared. Report these transfer-policy differences
when comparing accuracy, acceptance rates and bytes. Never infer quality from smoke PASS.

Output: `/kaggle/working/results_candle_v3_receiver_head_xi_0p8_seed_42/`.
Completion: `candle_v3_completion.json`; evaluation ZIP: `candle_v3_self_task_5.zip`.


In [ ]:
# CANDLE V3 receiver-encoder head learning; full engineering baseline.
import os
from pathlib import Path
import subprocess
import sys

CODE_REF = "e0565dd4c1a18c11d84007d0b60cb2bc9110dcee"
DATA_DIR = "/kaggle/input/datasets/khoilv2005/100-clients"
OUTPUT_DIR = "/kaggle/working/results_candle_v3_receiver_head_xi_0p8_seed_42"
# Point to mounted historical role_lock.json's DIRECTORY to reuse the exact split.
# An empty directory here recreates roles from original TRAIN using split seed20261006.
ROLES_DIR = "/kaggle/working/candle_v3_roles"

env = {**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"}
code = Path("/kaggle/temp/candle_v3_receiver_head_source")
code.parent.mkdir(parents=True, exist_ok=True)
if not code.exists():
    subprocess.check_call(["git", "clone", "--filter=blob:none", "--no-checkout",
        "https://github.com/khoilv2005/FL_IL_IDS.git", str(code)], env=env)
    subprocess.check_call(["git", "sparse-checkout", "set", "--no-cone",
        "/candle_v3_runtime/", "/tools/", "/configs/",
        "/requirements-candle-v3-kaggle.txt"], cwd=code, env=env)
    subprocess.check_call(["git", "checkout", "--detach", CODE_REF], cwd=code, env=env)
actual = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=code, text=True).strip()
if actual != CODE_REF:
    raise ValueError("Source differs from pinned CANDLE V3 revision; use a fresh session")
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r",
    str(code / "requirements-candle-v3-kaggle.txt")])
print("CANDLE V3 supporting source:", actual, flush=True)
subprocess.check_call([sys.executable, "-u", str(code / "tools/train_candle_v3.py"),
    "--peer-teaching", "--data-dir", DATA_DIR, "--output", OUTPUT_DIR, "--roles", ROLES_DIR], cwd=code, env=env)
print(Path(OUTPUT_DIR, "candle_v3_completion.json").read_text(encoding="utf-8"))
